# Notebook 02 — Data Cleaning
## China National Sword: Computing China Share & Pre/Post Flags

**This notebook:**
- Loads raw Comtrade data from `data/raw/comtrade_raw.csv`
- Computes China's share of World exports per commodity per year
- Adds pre/post National Sword period flags
- Calculates year-over-year changes
- Saves to `data/processed/comtrade_clean.csv`

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

RAW_FILE = Path('../data/raw/comtrade_raw.csv')
PROC_DIR = Path('../data/processed')
PROC_DIR.mkdir(parents=True, exist_ok=True)
OUT_FILE = PROC_DIR / 'comtrade_clean.csv'

df = pd.read_csv(RAW_FILE)
print(f'Loaded {len(df)} rows')
print(df.head())

## Step 1 — Pivot to wide format
One row per year/commodity, columns for China and World.

In [ ]:
# Separate China and World rows
china = df[df['partner_name'] == 'China'][['year','cmd_code','cmd_name','net_weight_mt','fob_value_usd']].copy()
world = df[df['partner_name'] == 'World'][['year','cmd_code','net_weight_mt','fob_value_usd']].copy()

china = china.rename(columns={'net_weight_mt':'china_mt', 'fob_value_usd':'china_usd'})
world = world.rename(columns={'net_weight_mt':'world_mt', 'fob_value_usd':'world_usd'})

wide = china.merge(world, on=['year','cmd_code'])
print(wide.shape)
print(wide.head())

## Step 2 — China share of World exports

In [ ]:
wide['china_share_pct'] = (wide['china_mt'] / wide['world_mt'].replace(0, np.nan) * 100).round(2)

# Quick check: waste paper share over time
paper = wide[wide['cmd_code'] == '4707'][['year','china_mt','world_mt','china_share_pct']]
print('Waste Paper — China share of World exports')
print(paper.to_string(index=False))

## Step 3 — Period flags & year-over-year changes

In [ ]:
def assign_period(year):
    if year <= 2017: return 'pre'
    elif year == 2018: return 'shock'
    else: return 'post'

wide['period'] = wide['year'].apply(assign_period)

# YoY change in China MT per commodity
wide = wide.sort_values(['cmd_code','year'])
wide['china_mt_yoy'] = wide.groupby('cmd_code')['china_mt'].pct_change() * 100
wide['china_share_yoy'] = wide.groupby('cmd_code')['china_share_pct'].diff()

print(wide[wide['cmd_code'] == '4707'][['year','period','china_mt','china_share_pct','china_mt_yoy']].to_string(index=False))

## Step 4 — Summary stats: pre vs post National Sword

In [ ]:
pre_avg = wide[wide['period']=='pre'].groupby('cmd_code')['china_mt'].mean().rename('pre_avg_mt')
post_avg = wide[wide['period']=='post'].groupby('cmd_code')['china_mt'].mean().rename('post_avg_mt')

summary = pd.concat([pre_avg, post_avg], axis=1)
summary['pct_change'] = ((summary['post_avg_mt'] - summary['pre_avg_mt']) / summary['pre_avg_mt'] * 100).round(1)

cmd_names = wide[['cmd_code','cmd_name']].drop_duplicates().set_index('cmd_code')
summary = summary.join(cmd_names)

print('Pre vs Post National Sword — China export volumes')
print(summary[['cmd_name','pre_avg_mt','post_avg_mt','pct_change']].to_string())

## Step 5 — Save cleaned data

In [ ]:
wide.to_csv(OUT_FILE, index=False)
print(f'Saved {len(wide)} rows -> {OUT_FILE}')
print(f'Columns: {list(wide.columns)}')
print(wide.dtypes)